# TP Lógico

Franco Martín González

---


In [1]:
import janus_swi as janus


## Lógica proposicional


* Representación de operadores lógicos como listas
* La negación se puede representar como una lista de dos elementos: la palabra neg y una expresión lógica.
* Las conjunciones, disyunciones e implicaciones se pueden representar como una lista de tres elementos: el alcance izquierdo, la conexión lógica y el alcance derecho.
* Conexiones binarias: and, or, then
* Constantes lógicas son expresiones: v,f
* Variables lógicas son expresiones: p, q, r y s.
* Si A es una expresión: [neg, A] también lo es.
* Si A y B son expresiones y Con es una conexión lógica: [A, Con, B] es una expresión.


In [2]:
janus.consult('Lógica proposicional', '''
% Representacion de formulas de logica proposicional como listas.
%   negacion:  [neg, A]
%   binarias:  [A, Con, B]  con Con en {and, or, then}

conbi(and).
conbi(or).
conbi(then).

% Constantes logicas.
clog(v).
clog(f).

% Variables proposicionales.
vlog(p).
vlog(q).
vlog(r).
vlog(s).

% expr(E): E es una formula bien formada.
expr(X) :- clog(X).
expr(X) :- vlog(X).
expr([neg, A]) :- expr(A).
expr([A, Con, B]) :- expr(A), conbi(Con), expr(B).

% --- Evaluacion de una formula dada una asignacion de valores ---
% valor(E, Asignacion, V): la formula E vale V bajo Asignacion,
% una lista de pares Variable-Valor. Los valores son v (verdadero) y f (falso).

valor(v, _, v).
valor(f, _, f).
valor(X, Asig, V) :- vlog(X), memberchk(X-V, Asig).
valor([neg, A], Asig, V) :- valor(A, Asig, VA), negado(VA, V).
valor([A, Con, B], Asig, V) :-
    valor(A, Asig, VA),
    valor(B, Asig, VB),
    tabla(Con, VA, VB, V).

negado(v, f).
negado(f, v).

% Tablas de verdad de las conexiones binarias.
tabla(and,  v, v, v).
tabla(and,  v, f, f).
tabla(and,  f, v, f).
tabla(and,  f, f, f).
tabla(or,   v, v, v).
tabla(or,   v, f, v).
tabla(or,   f, v, v).
tabla(or,   f, f, f).
tabla(then, v, v, v).
tabla(then, v, f, f).
tabla(then, f, v, v).
tabla(then, f, f, v).
''')


In [3]:
# p & (q -> ¬r) es una fórmula bien formada
list(janus.query('expr([p, and, [q, then, [neg, r]]]).'))


[{'truth': True}]

In [4]:
# evaluación bajo p=v, q=v, r=f
list(janus.query('valor([p, and, [q, then, [neg, r]]], [p-v, q-v, r-f], V).'))


[{'truth': True, 'V': 'v'}]

## fibonacci(N, X)


* Definir la relación fibonacci(N, X) que se verifique si X es el N-ésimo término de la sucesión de Fibonacci
* La sucesión de Fibonacci es 0, 1, 1, 2, 3, 5, 8, ...
* y está definida por
* f(0) = 0
* f(1) = 1
* f(n) = f(n-1) + f(n-2), si n > 1


In [5]:
janus.consult('fibonacci(N, X)', '''
% fibonacci(N, X): X es el N-esimo termino de la sucesion de Fibonacci.
% La version ingenua recalcula los mismos terminos y es exponencial,
% asi que uso un acumulador que arrastra el par (actual, siguiente):
% en cada paso avanza una posicion. Queda lineal en N.
fibonacci(N, X) :- fib_ac(N, 0, 1, X).

fib_ac(0, Actual, _, Actual).
fib_ac(N, Actual, Siguiente, X) :-
    N > 0,
    N1 is N - 1,
    Suma is Actual + Siguiente,
    fib_ac(N1, Siguiente, Suma, X).
''')


In [6]:
list(janus.query('fibonacci(9, X).'))


[{'truth': True, 'X': 34}]

In [7]:
# la versión lineal resuelve N grande sin problema
list(janus.query('fibonacci(30, X).'))


[{'truth': True, 'X': 832040}]

## Juego Torres de Hanoi


Programa en el que se tiene que mover todos los aros hacia el tercer poste, usando el segundo como un poste auxiliar. Para calcular los movimientos es mov=(2^n )-1, donde n es el # de aros


In [8]:
janus.consult('Juego Torres de Hanoi', '''
% Torres de Hanoi: mover N aros del poste Origen al Destino usando Auxiliar.
% Estrategia clasica: mover N-1 aros al auxiliar, pasar el aro grande al
% destino, y mover los N-1 del auxiliar al destino.
hanoi(1, Origen, _, Destino) :-
    mostrar_movimiento(Origen, Destino).
hanoi(N, Origen, Auxiliar, Destino) :-
    N > 1,
    M is N - 1,
    hanoi(M, Origen, Destino, Auxiliar),
    mostrar_movimiento(Origen, Destino),
    hanoi(M, Auxiliar, Origen, Destino).

mostrar_movimiento(Origen, Destino) :-
    format("Mueve del ~w al ~w~n", [Origen, Destino]).

% cantidad_movimientos(N, Cant): 2^N - 1, la formula del enunciado.
cantidad_movimientos(N, Cant) :- Cant is 2**N - 1.
''')


In [9]:
list(janus.query('hanoi(3, a, b, c).'))


Mueve del a al c
Mueve del a al b
Mueve del c al b
Mueve del a al c
Mueve del b al a
Mueve del b al c
Mueve del a al c


[{'truth': True}]

In [10]:
list(janus.query('cantidad_movimientos(3, C).'))


[{'truth': True, 'C': 7}]

## Rutas


Rutas que permite encontrar la distancia en kilómetros entre dos ciudades pasando por diferentes rutas


In [11]:
janus.consult('Rutas', '''
% Distancia en km entre ciudades, posiblemente pasando por escalas.
distancia(buenosAires, puertoMadryn, 1300).
distancia(puertoMadryn, puertoDeseado, 732).
distancia(puertoDeseado, rioGallegos, 736).
distancia(puertoDeseado, calafate, 979).
distancia(rioGallegos, calafate, 304).
distancia(calafate, chalten, 213).

% Los tramos son bidireccionales: se puede viajar en cualquier sentido.
tramo(A, B, Kms) :- distancia(A, B, Kms).
tramo(A, B, Kms) :- distancia(B, A, Kms).

% Llevo la lista de ciudades ya visitadas para no entrar en ciclos:
% sin eso, ir y volver por el mismo tramo daria recursion infinita.
viaje(Origen, Destino, Kms) :- viaje(Origen, Destino, [Origen], Kms).

viaje(Destino, Destino, _, 0).
viaje(Origen, Destino, Visitadas, Kms) :-
    tramo(Origen, Intermedia, KmsTramo),
    \+ memberchk(Intermedia, Visitadas),
    viaje(Intermedia, Destino, [Intermedia|Visitadas], KmsResto),
    Kms is KmsTramo + KmsResto.
''')


<>:21: SyntaxWarning: invalid escape sequence '\+'
<>:21: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_314483/1483508639.py:21: SyntaxWarning: invalid escape sequence '\+'
  \+ memberchk(Intermedia, Visitadas),


In [12]:
list(janus.query('viaje(buenosAires, chalten, Kms).'))


[{'truth': True, 'Kms': 3285}, {'truth': True, 'Kms': 3224}]

In [13]:
# simétrico: también resuelve la vuelta
list(janus.query('viaje(chalten, buenosAires, Kms).'))


[{'truth': True, 'Kms': 3224}, {'truth': True, 'Kms': 3285}]

## cons(X, L1, L2)


Definir la relación cons(X, L1, L2) que se verifique si L2 es la lista obtenida añadiendo X a L1 como primer elemento


In [14]:
janus.consult('cons(X, L1, L2)', '''
% cons(X, L1, L2): L2 es L1 con X agregado como primer elemento.
% Un unico hecho alcanza: la relacion es pura unificacion estructural.
cons(X, L1, [X|L1]).
''')


In [15]:
list(janus.query('cons(a, [b, c], L).'))


[{'truth': True, 'L': ['a', 'b', 'c']}]

## inversa(L1, L2)


Definir la relación inversa(L1, L2) que se verifique si L2 es la lista obtenida invirtiendo el orden de los elementos de la lista L1.


In [16]:
janus.consult('inversa(L1, L2)', '''
% inversa(L1, L2): L2 es L1 con el orden invertido.
% Uso un acumulador: voy pasando la cabeza al frente del acumulador,
% asi la recursion es de cola y evito el append/3 cuadratico.
inversa(L1, L2) :- inversa_ac(L1, [], L2).

inversa_ac([], Ac, Ac).
inversa_ac([X|Resto], Ac, L2) :- inversa_ac(Resto, [X|Ac], L2).
''')


In [17]:
list(janus.query('inversa([1, 2, 3], L).'))


[{'truth': True, 'L': [3, 2, 1]}]

## palindromo(L)


Definir la relación palíndromo(L) que se verifique si la lista L es un palíndromo


In [18]:
janus.consult('palindromo(L)', '''
% palindromo(L): L se lee igual en ambos sentidos.
% Defino la inversion aca mismo (con nombre propio para no pisar
% la del ejercicio 6) para que el archivo sea autocontenido.
palindromo(L) :- invertir_ac(L, [], L).

invertir_ac([], Ac, Ac).
invertir_ac([X|Resto], Ac, Inv) :- invertir_ac(Resto, [X|Ac], Inv).
''')


In [19]:
list(janus.query('palindromo([a, b, a]).'))


[{'truth': True}]

In [20]:
list(janus.query('palindromo([a, b]).'))


[]

## selecciona(X, L1, L2)


Definir la relación selecciona(X, L1, L2) que se verifique si L2 es la lista obtenida eliminando una ocurrencia de X en L1


In [21]:
janus.consult('selecciona(X, L1, L2)', '''
% selecciona(X, L1, L2): L2 es L1 sin UNA ocurrencia de X.
% Caso base: X esta en la cabeza y lo salteo, la cola queda igual.
% Caso recursivo: conservo la cabeza Y y sigo buscando en el resto.
selecciona(X, [X|L], L).
selecciona(X, [Y|L1], [Y|L2]) :- selecciona(X, L1, L2).
''')


In [22]:
list(janus.query('selecciona(b, [a, b, c], L).'))


[{'truth': True, 'L': ['a', 'c']}]

In [23]:
# con repetidos, una solución por ocurrencia
list(janus.query('selecciona(a, [a, b, a], L).'))


[{'truth': True, 'L': ['b', 'a']}, {'truth': True, 'L': ['a', 'b']}]

## inserta(X, L1, L2)


Definir la relación inserta(X, L1, L2) que se verifique si L2 es una lista obtenida insertando X en L1


In [24]:
janus.consult('inserta(X, L1, L2)', '''
% inserta(X, L1, L2): L2 es L1 con X insertado en alguna posicion.
% Es la relacion inversa de selecciona/3: insertar X en L1 dando L2
% equivale a sacar X de L2 dando L1. Redefino la extraccion aca
% para que el archivo funcione por si solo.
inserta(X, L1, L2) :- extraer(X, L2, L1).

extraer(X, [X|L], L).
extraer(X, [Y|L1], [Y|L2]) :- extraer(X, L1, L2).
''')


In [25]:
list(janus.query('inserta(x, [a, b], L).'))


[{'truth': True, 'L': ['x', 'a', 'b']},
 {'truth': True, 'L': ['a', 'x', 'b']},
 {'truth': True, 'L': ['a', 'b', 'x']}]

## todos_iguales(L)


Definir la relación todos_iguales(L) que se verifique si todos los elementos de la lista L son iguales entre sí


In [26]:
janus.consult('todos_iguales(L)', '''
% todos_iguales(L): todos los elementos de L son iguales entre si.
% Comparo de a pares consecutivos: si los dos primeros unifican,
% sigo con la cola. La lista vacia y la unitaria son casos base.
todos_iguales([]).
todos_iguales([_]).
todos_iguales([X, X|Resto]) :- todos_iguales([X|Resto]).
''')


In [27]:
list(janus.query('todos_iguales([a, a, a]).'))


[{'truth': True}]

In [28]:
list(janus.query('todos_iguales([a, b]).'))


[]

## longitud_par(L)


Definir la relación longitud_par(L) que se verifique si la longitud de la lista L es par


In [29]:
janus.consult('longitud_par(L)', '''
% longitud_par(L): la longitud de L es par.
% Dos predicados mutuamente recursivos: sacar un elemento cambia
% la paridad, asi que par delega en impar y viceversa.
% No uso length/2 para que sea puramente recursivo.
%
% Las clausulas son disjuntas ([] contra [_|_]) para que cada lista
% tenga UNA sola solucion y no haya respuestas repetidas.
longitud_par([]).
longitud_par([_|Resto]) :- longitud_impar(Resto).

longitud_impar([_|Resto]) :- longitud_par(Resto).
''')


In [30]:
list(janus.query('longitud_par([a, b]).'))


[{'truth': True}]

In [31]:
list(janus.query('longitud_par([a, b, c]).'))


[]

## subconjunto(L1, L2)


Definir la relación subconjunto(L1, L2) que se verifique si L2 es un subconjunto de L1


In [32]:
janus.consult('subconjunto(L1, L2)', '''
% subconjunto(L1, L2): L2 es un subconjunto de L1.
% Para cada elemento de L1 hay dos opciones: incluirlo en L2 o descartarlo.
% Cuando L1 se agota, L2 tiene que estar cerrada en [].
subconjunto([], []).
subconjunto([X|L1], [X|L2]) :- subconjunto(L1, L2).
subconjunto([_|L1], L2) :- subconjunto(L1, L2).
''')


In [33]:
list(janus.query('subconjunto([a, b, c], L).'))


[{'truth': True, 'L': ['a', 'b', 'c']},
 {'truth': True, 'L': ['a', 'b']},
 {'truth': True, 'L': ['a', 'c']},
 {'truth': True, 'L': ['a']},
 {'truth': True, 'L': ['b', 'c']},
 {'truth': True, 'L': ['b']},
 {'truth': True, 'L': ['c']},
 {'truth': True, 'L': []}]

## lista_acotada(L)


Definir la relación lista_acotada(L) que se verifique si todos los elementos de la lista de números L son menores que la longitud de L


In [34]:
janus.consult('lista_acotada(L)', '''
% lista_acotada(L): todo elemento de L es menor que la longitud de L.
% Calculo la longitud una sola vez y la paso como cota fija.
lista_acotada(L) :- length(L, N), menores_que(L, N).

menores_que([], _).
menores_que([X|Resto], N) :- X < N, menores_que(Resto, N).
''')


In [35]:
list(janus.query('lista_acotada([0, 1, 2]).'))


[{'truth': True}]

In [36]:
list(janus.query('lista_acotada([0, 5]).'))


[]

## max_lista(L, X)


Definir la relación max_lista(L, X) que se verifique si X es el máximo de la lista de números L


In [37]:
janus.consult('max_lista(L, X)', '''
% max_lista(L, X): X es el maximo de la lista de numeros L.
% Recorro con acumulador: comparo el candidato actual contra la cabeza.
max_lista([X|Resto], Max) :- max_ac(Resto, X, Max).

max_ac([], Max, Max).
max_ac([X|Resto], Parcial, Max) :-
    (   X > Parcial
    ->  max_ac(Resto, X, Max)
    ;   max_ac(Resto, Parcial, Max)
    ).
''')


In [38]:
list(janus.query('max_lista([3, 7, 2], X).'))


[{'truth': True, 'X': 7}]

In [39]:
# también con negativos
list(janus.query('max_lista([-3, -7], X).'))


[{'truth': True, 'X': -3}]

## ordenada(L)


Definir la relación ordenada(L) que se verifique si la lista de números está ordenada de manera creciente


In [40]:
janus.consult('ordenada(L)', '''
% ordenada(L): la lista de numeros L esta ordenada de forma creciente.
% Comparo pares consecutivos. Incluyo [] como ordenada.
ordenada([]).
ordenada([_]).
ordenada([X, Y|Resto]) :- X =< Y, ordenada([Y|Resto]).
''')


In [41]:
list(janus.query('ordenada([1, 2, 2, 5]).'))


[{'truth': True}]

In [42]:
list(janus.query('ordenada([3, 1]).'))


[]

## lista(N, L)


Definir la relación lista(N, L) que se verifique si L es la lista de longitud N cuyos elementos son N


In [43]:
janus.consult('lista(N, L)', '''
% lista(N, L): L es la lista de longitud N cuyos elementos son todos N.
% Uso un contador que baja de N a 0; N se mantiene fijo como valor a repetir.
lista(N, L) :- repetir(N, N, L).

repetir(_, 0, []).
repetir(N, Cont, [N|Resto]) :-
    Cont > 0,
    Cont1 is Cont - 1,
    repetir(N, Cont1, Resto).
''')


In [44]:
list(janus.query('lista(3, L).'))


[{'truth': True, 'L': [3, 3, 3]}]

## entre(N1, N2, X)


Definir la relación entre(N1, N2, X) que se verifique si X es un número entero tal que N1 ≤ X ≤ N2


In [45]:
janus.consult('entre(N1, N2, X)', '''
% entre(N1, N2, X): X es un entero tal que N1 =< X =< N2.
% Por backtracking genera todos los valores del intervalo, uno por solucion.
entre(N1, N2, N1) :- N1 =< N2.
entre(N1, N2, X) :-
    N1 < N2,
    Siguiente is N1 + 1,
    entre(Siguiente, N2, X).
''')


In [46]:
list(janus.query('entre(2, 5, X).'))


[{'truth': True, 'X': 2},
 {'truth': True, 'X': 3},
 {'truth': True, 'X': 4},
 {'truth': True, 'X': 5}]

## elemento_en(K, L, X)


Definir la relación elemento_en(K, L, X) que se verifique si X es el K-ésimo elemento de la lista L(se empieza a numerar en 1)


In [47]:
janus.consult('elemento_en(K, L, X)', '''
% elemento_en(K, L, X): X es el K-esimo elemento de L (numerando desde 1).
% Bajo K de a uno mientras descarto cabezas, hasta llegar a 1.
elemento_en(1, [X|_], X).
elemento_en(K, [_|Resto], X) :-
    K > 1,
    K1 is K - 1,
    elemento_en(K1, Resto, X).
''')


In [48]:
list(janus.query('elemento_en(2, [a, b, c], X).'))


[{'truth': True, 'X': 'b'}]

## multiplicada(L1, N, L2)


Definir la relación multiplicada(L1, N, L2) que se verifica si L2 es la lista obtenida repitiendo N veces los elementos de la lista L1


In [49]:
janus.consult('multiplicada(L1, N, L2)', '''
% multiplicada(L1, N, L2): L2 repite N veces cada elemento de L1.
% Para cada elemento genero sus N copias y concateno con el resto.
multiplicada([], _, []).
multiplicada([X|Resto], N, L2) :-
    copias(X, N, Copias),
    multiplicada(Resto, N, RestoMult),
    append(Copias, RestoMult, L2).

copias(_, 0, []).
copias(X, N, [X|Resto]) :-
    N > 0,
    N1 is N - 1,
    copias(X, N1, Resto).
''')


In [50]:
list(janus.query('multiplicada([a, b], 3, L).'))


[{'truth': True, 'L': ['a', 'a', 'a', 'b', 'b', 'b']}]

## Familia


In [51]:
janus.consult('Familia', '''
% Relaciones de parentesco a partir del hecho basico padre/2.
padre(alberto, juan).
padre(alberto, luis).
padre(leoncio, alberto).
padre(leoncio, geronimo).
padre(geronimo, luisa).

hijo(H, P) :- padre(P, H).

% Hermanos: comparten padre pero no son la misma persona.
hermano(A, B) :- padre(P, A), padre(P, B), A \== B.

nieto(N, A) :- padre(P, N), padre(A, P).
abuelo(A, N) :- nieto(N, A).

% Tio: hermano de alguno de los padres.
tio(T, S) :- padre(P, S), hermano(P, T).

% Primos: sus padres son hermanos entre si.
primo(A, B) :- padre(PA, A), padre(PB, B), hermano(PA, PB).

% Antepasado: padre, o padre de un antepasado (clausura transitiva).
antepasado(A, D) :- padre(A, D).
antepasado(A, D) :- padre(A, X), antepasado(X, D).
''')


<>:12: SyntaxWarning: invalid escape sequence '\='
<>:12: SyntaxWarning: invalid escape sequence '\='
/tmp/ipykernel_314483/1136051830.py:12: SyntaxWarning: invalid escape sequence '\='
  hermano(A, B) :- padre(P, A), padre(P, B), A \== B.


In [52]:
list(janus.query('hermano(A, B).'))


[{'truth': True, 'A': 'juan', 'B': 'luis'},
 {'truth': True, 'A': 'luis', 'B': 'juan'},
 {'truth': True, 'A': 'alberto', 'B': 'geronimo'},
 {'truth': True, 'A': 'geronimo', 'B': 'alberto'}]

In [53]:
list(janus.query('tio(T, S).'))


[{'truth': True, 'T': 'geronimo', 'S': 'juan'},
 {'truth': True, 'T': 'geronimo', 'S': 'luis'},
 {'truth': True, 'T': 'alberto', 'S': 'luisa'}]

In [54]:
# clausura transitiva: descendientes de leoncio
list(janus.query('antepasado(leoncio, X).'))


[{'truth': True, 'X': 'alberto'},
 {'truth': True, 'X': 'geronimo'},
 {'truth': True, 'X': 'juan'},
 {'truth': True, 'X': 'luis'},
 {'truth': True, 'X': 'luisa'}]

## Películas


* Un suertude es aquel que actuó en una película que ganó el Oscar
* p ^ q -> r
* p: actuó en una película
* q: ganó el Oscar
* r: es suertudo


In [55]:
janus.consult('Películas', '''
% Un suertudo es quien actuo en una pelicula que gano el Oscar.
% Formalizacion: actuo(P, Peli) ^ ganoElOscar(Peli) -> suertudo(P)
actuo(leoDiCaprio, wolfOfWallStreet).
actuo(margotRobbie, wolfOfWallStreet).
actuo(jonahHill, wolfOfWallStreet).
actuo(leoDiCaprio, onceUponATimeInHollywood).
actuo(bradPitt, onceUponATimeInHollywood).
actuo(margotRobbie, onceUponATimeInHollywood).
actuo(joePesci, goodFellas).
actuo(robertDeNiro, goodFellas).
actuo(rayLiotta, goodFellas).
actuo(lorraineBracco, goodFellas).
actuo(leoDiCaprio, catchMeIfYouCan).
actuo(tomHanks, catchMeIfYouCan).
actuo(michaelKeaton, birdman).
actuo(emmaStone, birdman).

ganoElOscar(birdman).

suertudo(Persona) :- actuo(Persona, Pelicula), ganoElOscar(Pelicula).

% Companeros de reparto en alguna pelicula.
companeros(A, B) :- actuo(A, Peli), actuo(B, Peli), A \== B.
''')


<>:24: SyntaxWarning: invalid escape sequence '\='
<>:24: SyntaxWarning: invalid escape sequence '\='
/tmp/ipykernel_314483/1927435053.py:24: SyntaxWarning: invalid escape sequence '\='
  companeros(A, B) :- actuo(A, Peli), actuo(B, Peli), A \== B.


In [56]:
list(janus.query('suertudo(P).'))


[{'truth': True, 'P': 'michaelKeaton'}, {'truth': True, 'P': 'emmaStone'}]

## Buen trabajo


* Una persona tiene un buen trabajo si
    * gana un buen sueldo (> 800K)
    * tiene tiempo libre para vivir (trabaja <= 8 horas)


In [57]:
janus.consult('Buen trabajo', '''
% Tiene un buen trabajo quien gana buen sueldo (> 800000)
% y ademas tiene tiempo libre (trabaja <= 8 horas).
sueldo(elektra, 850000).
sueldo(loki, 900000).
sueldo(thor, 750000).

trabaja(elektra, 6).
trabaja(thor, 5).
trabaja(loki, 8).

ganaBuenSueldo(Quien) :- sueldo(Quien, Monto), Monto > 800000.

tieneTiempoLibre(Quien) :- trabaja(Quien, Horas), Horas =< 8.

tieneUnBuenTrabajo(Quien) :- ganaBuenSueldo(Quien), tieneTiempoLibre(Quien).
''')


In [58]:
list(janus.query('tieneUnBuenTrabajo(Q).'))


[{'truth': True, 'Q': 'elektra'}, {'truth': True, 'Q': 'loki'}]

## Caminos


* El problema consiste en determinar caminos entre ciudades
* Hay un camino entre dos ciudades si hay una ruta que las une
* Además, hay un camino entre una ciudad x y una ciudad z, si hay una ruta que une a x con otra ciudad t, seguida de un camino de t a z
* Dadas 6 ciudades ba, lp, mdq, jy, mi y bb, se sabe que hay ruta entre: ba y lp; ba y jy; mi y bb; ba y mdq; ba y mi; mdq y mi; lp y mi
* Se pide contestar: ¿Existe algún camino a la ciudad mdq?. Si existe, ¿desde que ciudad/es?


In [59]:
janus.consult('Caminos', '''
% Caminos entre ciudades. Hay camino si hay una ruta directa, o una ruta
% a una ciudad intermedia seguida de un camino desde alli.
ruta(ba, lp).
ruta(ba, jy).
ruta(mi, bb).
ruta(ba, mdq).
ruta(ba, mi).
ruta(mdq, mi).
ruta(lp, mi).

% Las rutas se pueden recorrer en ambos sentidos.
conectada(A, B) :- ruta(A, B).
conectada(A, B) :- ruta(B, A).

% Acumulo visitadas para cortar los ciclos que genera la simetria.
camino(Origen, Destino) :- camino(Origen, Destino, [Origen]).

camino(Origen, Destino, _) :- conectada(Origen, Destino).
camino(Origen, Destino, Visitadas) :-
    conectada(Origen, Intermedia),
    \+ memberchk(Intermedia, Visitadas),
    camino(Intermedia, Destino, [Intermedia|Visitadas]).
''')


<>:22: SyntaxWarning: invalid escape sequence '\+'
<>:22: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_314483/3588317665.py:22: SyntaxWarning: invalid escape sequence '\+'
  \+ memberchk(Intermedia, Visitadas),


In [60]:
# la pregunta del enunciado: desde qué ciudades hay camino a mdq
list(janus.query('camino(X, mdq).'))


[{'truth': True, 'X': 'ba'},
 {'truth': True, 'X': 'mi'},
 {'truth': True, 'X': 'ba'},
 {'truth': True, 'X': 'ba'},
 {'truth': True, 'X': 'ba'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'lp'},
 {'truth': True, 'X': 'jy'},
 {'truth': True, 'X': 'jy'},
 {'truth': True, 'X': 'jy'},
 {'truth': True, 'X': 'jy'},
 {'truth': True, 'X': 'bb'},
 {'truth': True, 'X': 'bb'},
 {'truth': True, 'X': 'bb'},
 {'truth': True, 'X': 'bb'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'mdq'},
 {'truth': True, 'X': 'mi'},
 {'truth': True, 'X': 'mi'},
 {'truth': True, 'X': 'mi'}]

## Autómatas


* Representar un AFD (autómata finito determinista) que reconoce el lenguaje {b, bb, ab, abb, aab, aabb, aaab, aaabb, ...} es decir a*(b | bb) utilizando las siguientes relaciones: 
    * final(Q) que se verifica si Q es el estado final
    * trans(Q1, S, Q2) que se verifica si se puede pasar del estado Q1 al estado Q2 leyendo el símbolo S
* Definir la relación acepta(L, Q) que se verifica si el autómata, a partir del estado Q, acepta la lista L (cadena de símbolos)


In [61]:
janus.consult('Autómatas', '''
% AFD que reconoce el lenguaje a*(b | bb), es decir:
% {b, bb, ab, abb, aab, aabb, aaab, aaabb, ...}
%
%   q0 --a--> q0     (cualquier cantidad de aes, incluida ninguna)
%   q0 --b--> q1     (primera b: ya es aceptable)
%   q1 --b--> q2     (segunda b: tambien aceptable)
%
% q0 no es final porque la cadena vacia y "aaa" no pertenecen al lenguaje.
inicial(q0).

final(q1).
final(q2).

trans(q0, a, q0).
trans(q0, b, q1).
trans(q1, b, q2).

% acepta(L, Q): desde el estado Q, el automata acepta la cadena L.
% Consumida toda la entrada, hay que estar parado en un estado final.
acepta([], Q) :- final(Q).
acepta([Simbolo|Resto], Q) :-
    trans(Q, Simbolo, Q2),
    acepta(Resto, Q2).

% Punto de entrada: acepta desde el estado inicial.
acepta(L) :- inicial(Q0), acepta(L, Q0).
''')


In [62]:
# pertenece al lenguaje a*(b|bb)
list(janus.query('acepta([a, a, b, b]).'))


[{'truth': True}]

In [63]:
# no pertenece
list(janus.query('acepta([a, b, a]).'))


[]